# UX Frustration Recognition Experiment

In [ ]:
%matplotlib widget
from IPython.display import display
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime
import joblib

from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline
from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

STD_BY_SUBJ = True
task_training = "VidTasks"
log_prefix = "BalOvSamp"
MODELS_DATE = "20251017"

std_suffix = ".stdsubj" if STD_BY_SUBJ else ".stdtask"
std_models = f"StdSubject_{task_training}" if STD_BY_SUBJ else "StdTask"

BASE_PATH = "../../experiment-data"
LABELS_SEPARATOR = ","
LABELS_FILENAME = "labels.csv"
FEATURES_SEPARATOR = ";"
FEATURES_DIRECTORY = f"{BASE_PATH}/extracted-features"
FEATURES_FILENAME = f"{FEATURES_DIRECTORY}/all_features{std_suffix}.csv"
MODELS_PATH = f"Results_{std_models}/{MODELS_DATE}_{log_prefix}"
RESULTS_SEPARATOR = ","
RESULTS_DIRECTORY = f"{BASE_PATH}/results/{task_training}"
BIN_LABELS = ["NoStress", "Stress"]
TER_LABELS = ["Relaxed", "Stress", "RealStress"]
QAD_LABELS = ["Relaxed", "Stress", "RealStress", "Amused"]
DEBUG = True
FLOAT_TYPE = np.float64

STRESS_TASKS = ["Baseline", "AmusementClip", "StressClip"]
FRUSTRATION_TASKS = ["EmoReset", "FormL", "FormM"]

### Import labels and dataset

In [ ]:
labels = pd.read_csv(f"{BASE_PATH}/{LABELS_FILENAME}", sep=LABELS_SEPARATOR, header=0, index_col=0).dropna()
display(labels)

In [ ]:
X = pd.read_csv(f"{FEATURES_FILENAME}", sep=FEATURES_SEPARATOR, header=0, index_col=0)
display(X)
for col in X.columns:
    X[col] = X[col].astype(FLOAT_TYPE)
display(X)


In [ ]:
# Selecting rows that actually have entries in "labels" file
idx = list(X.merge(labels, left_index=True, right_index=True).index)
labels = labels.loc[idx]
x = X.loc[idx]
# Debriefing task has no questionnaire data, it could be used as unseen data and labelled as expected "relax" without ground truth
print(f"Selected {len(x)} entries from X. Not considering {len(X) - len(x)} entries.")

In [ ]:
stress_mask = [stress_file.split("-")[1] in STRESS_TASKS for stress_file in labels.index]
labels_stress = labels.loc[stress_mask]
frust_mask = [frust_file.split("-")[1] in FRUSTRATION_TASKS for frust_file in labels.index]
labels_frust = labels.loc[frust_mask]
print(f"Labels divided into {len(labels_stress)} stress tasks labels and {len(labels_frust)} frustration tasks labels")

x_stress = x.loc[stress_mask]
x_frust = x.loc[frust_mask]
print(f"Data divided into {len(x_stress)} stress tasks samples and {len(x_frust)} frustration tasks samples")

# Visualise class distribution
plt.figure(figsize=(12, 6))
y = labels["binary-stress"]
bin_stats = pd.DataFrame(
    {
        "Value": y.values,
        "Subject": [idx.split("-")[0] for idx in y.index],
        "Task": [idx.split("-")[1] for idx in y.index],
    },
    index=y.index,
)
for cls_id, cls_name in enumerate(BIN_LABELS):
    bin_stats[cls_name] = [value == cls_id for value in y]

tasks = sorted(bin_stats['Task'].unique())
x_positions = np.arange(len(tasks))
width = 0.35

# Plot bars and scatter points for each class
for i, class_name in enumerate(BIN_LABELS):
    counts = bin_stats.groupby('Task')[class_name].sum()
    bars = plt.bar(x_positions + (i-0.5)*width, [counts[task] for task in tasks], width,
                    label=class_name, alpha=0.7)

plt.xlabel('Task')
plt.ylabel('Count')
plt.title('Number of instances per class by Task')
plt.xticks(x_positions, tasks, rotation=45)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nInstances count per task:")
display(bin_stats.groupby('Task')[BIN_LABELS].sum())


### Helper functions

In [ ]:
def make_results_filename(classification: str, feature_selection: str | None) -> str:
    now = datetime.now()
    timestamp = now.strftime("%Y%m%d")
    feature_selection = feature_selection if feature_selection is not None else "None"
    return f"{RESULTS_DIRECTORY}/{timestamp}-{classification}_{feature_selection}f"


def show_label_stats(labels: list[str], y: pd.Series):
    display(
        pd.DataFrame(
            {
                "labels": labels,
                "counts": y.value_counts().to_list(),
                "percentage": (y.value_counts(normalize=True) * 100).to_list(),
            }
        )
    )

def show_results(classification: str, feature_selector: str, results: pd.DataFrame, save=False):
    display(results)
    if save:
        res_file = make_results_filename(classification, feature_selector) + ".csv"
        print(f"Saving results to {res_file}")
        results.to_csv(res_file, sep=RESULTS_SEPARATOR, index=True)

def show_confusion_matrices(classification: str, feature_selector: str, conf_matrices: dict[str, np.ndarray], labels: list[int], save=False):
    cm_file = make_results_filename(classification, feature_selector) + ".CM.png"
    ncms = len(conf_matrices)
    pairs = [(name, cm) for name, cm in conf_matrices.items()]
    nrows = (ncms // 2) + (ncms % 2)
    fig, axes = plt.subplots(nrows, 2, figsize=(8, nrows * 4))
    for (name, cm), ax in zip(pairs, axes.ravel()):
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
        disp.plot(cmap=plt.cm.Blues, ax=ax, colorbar=False)
        ax.set_title(name)
    fig.suptitle("Confusion Matrices")
    plt.tight_layout()
    if save:
        print(f"Saving confusion matrix to {cm_file}")
        plt.savefig(cm_file, dpi=300, format="png")
    plt.show()



### Parameters

In [ ]:
params = {
    "show_confusion_matrices": True,
    "save_confusion_matrices": True,
    "save_results": True,
    "verbose": False,
}

### Classification

In [ ]:
feat_sel = "RFE"
models_paths = {
    "b": {
        "KNeighborsClassifier": f"{MODELS_PATH}/{MODELS_DATE}-120143_binary_KNeighborsClassifier_{feat_sel}f.joblib",
        "MLP": f"{MODELS_PATH}/{MODELS_DATE}-120143_binary_MLPClassifier_HLs(100)_{feat_sel}f.joblib",
        "RandomForest": f"{MODELS_PATH}/{MODELS_DATE}-120143_binary_RandomForestClassifier_{feat_sel}f.joblib",
        "SVC": f"{MODELS_PATH}/{MODELS_DATE}-120143_binary_SVC_{feat_sel}f.joblib",
    },
    "t": {
        "KNeighborsClassifier": f"{MODELS_PATH}/{MODELS_DATE}-120820_ternary_KNeighborsClassifier_{feat_sel}f.joblib",
        "MLP": f"{MODELS_PATH}/{MODELS_DATE}-120820_ternary_MLPClassifier_HLs(100)_{feat_sel}f.joblib",
        "RandomForest": f"{MODELS_PATH}/{MODELS_DATE}-120820_ternary_RandomForestClassifier_{feat_sel}f.joblib",
        "SVC": f"{MODELS_PATH}/{MODELS_DATE}-120820_ternary_SVC_{feat_sel}f.joblib",
    },
    "q": {
        "KNeighborsClassifier": f"{MODELS_PATH}/{MODELS_DATE}-121222_quaternary_KNeighborsClassifier_{feat_sel}f.joblib",
        "MLP": f"{MODELS_PATH}/{MODELS_DATE}-121222_quaternary_MLPClassifier_HLs(100)_{feat_sel}f.joblib",
        "RandomForest": f"{MODELS_PATH}/{MODELS_DATE}-121221_quaternary_RandomForestClassifier_{feat_sel}f.joblib",
        "SVC": f"{MODELS_PATH}/{MODELS_DATE}-121222_quaternary_SVC_{feat_sel}f.joblib",
    },
}
models: dict[str, dict[str, Pipeline]] = {}
for class_type, paths in models_paths.items():
    models[class_type] = {}
    for model_type, path in paths.items():
        models[class_type][model_type] = joblib.load(path)

for class_type, models in models.items():
    if class_type == "b":
        params["classification"] = "binary"
        params["classes"] = BIN_LABELS
        y_stress = labels_stress["binary-stress"]
        y_frust = labels_frust["binary-stress"]
    elif class_type == "t":
        params["classification"] = "ternary"
        params["classes"] = TER_LABELS
        y_stress = labels_stress["affect3-class"]
        y_frust = labels_frust["affect3-class"]
    elif class_type == "q":
        params["classification"] = "quaternary"
        params["classes"] = QAD_LABELS
        y_stress = labels_stress["affect4-class"]
        y_frust = labels_frust["affect4-class"]

    print(f"\n\n============ {params['classification']} ============\n\n")
    print("Stress tasks (videoclips)")
    show_label_stats(params["classes"], y_stress)

    log_type = f"{log_prefix}_{params['classification']}"
    df_res = pd.DataFrame(
        {
            "classifier": [],
            "bal-accuracy": [],
            "weighted-f1": [],
            "macro-avg-prec": [],
            "macro-avg-rec": [],
            "macro-avg-f1": [],
            "stress-prec": [],
            "stress-rec": [],
            "stress-f1": [],
        }
    )
    conf_matrices: dict[str, np.ndarray] = {}

    for model_name, model in models.items():
        y_pred = model.predict(x_stress)
        cm = confusion_matrix(y_stress, y_pred)
        report = classification_report(y_stress, y_pred, target_names=params["classes"], output_dict=True)
        conf_matrices[model_name] = cm

        s_report = report["Stress"]
        m_report = report["macro avg"]
        new_row = {
            "classifier": model_name,
            "bal-accuracy": balanced_accuracy_score(y_stress, y_pred),
            "weighted-f1": f1_score(y_stress, y_pred, average="weighted"),
            "macro-avg-prec": m_report["precision"],
            "macro-avg-rec": m_report["recall"],
            "macro-avg-f1": m_report["f1-score"],
            "stress-prec": s_report["precision"],
            "stress-rec": s_report["recall"],
            "stress-f1": s_report["f1-score"],
        }
        df_res.loc[len(df_res)] = new_row

    show_results(log_type + "_VTasks", feat_sel, df_res, save=params["save_results"])
    if params["show_confusion_matrices"]:
        show_confusion_matrices(log_type + "_VTasks", feat_sel, conf_matrices, labels=params["classes"], save=params["save_confusion_matrices"])

    print("\nFrustration tasks (calculator)")
    show_label_stats(params["classes"], y_frust)

    df_res = pd.DataFrame(
        {
            "classifier": [],
            "bal-accuracy": [],
            "weighted-f1": [],
            "macro-avg-prec": [],
            "macro-avg-rec": [],
            "macro-avg-f1": [],
            "stress-prec": [],
            "stress-rec": [],
            "stress-f1": [],
        }
    )
    conf_matrices: dict[str, np.ndarray] = {}

    for model_name, model in models.items():
        y_pred = model.predict(x_frust)
        cm = confusion_matrix(y_frust, y_pred)
        report = classification_report(y_frust, y_pred, target_names=params["classes"], output_dict=True)
        conf_matrices[model_name] = cm

        s_report = report["Stress"]
        m_report = report["macro avg"]
        new_row = {
            "classifier": model_name,
            "bal-accuracy": balanced_accuracy_score(y_frust, y_pred),
            "weighted-f1": f1_score(y_frust, y_pred, average="weighted"),
            "macro-avg-prec": m_report["precision"],
            "macro-avg-rec": m_report["recall"],
            "macro-avg-f1": m_report["f1-score"],
            "stress-prec": s_report["precision"],
            "stress-rec": s_report["recall"],
            "stress-f1": s_report["f1-score"],
        }
        df_res.loc[len(df_res)] = new_row

    show_results(log_type + "_CTasks", feat_sel, df_res, save=params["save_results"])
    if params["show_confusion_matrices"]:
        show_confusion_matrices(
            log_type + "_CTasks",
            feat_sel,
            conf_matrices,
            labels=params["classes"],
            save=params["save_confusion_matrices"],
        )